In [ ]:
import itertools
from collections import defaultdict
from pathlib import Path

import equinox as eqx
import grain
import jax
import jax.numpy as jnp
import numpy as np
from context_flux_no.data.sources import ZarrWellDatasetSource
from context_flux_no.metrics import relative_L2_error, relative_L_infty_error
from context_flux_no.training.io import load_model
from tqdm import tqdm


jax.config.update("jax_default_device", jax.devices("gpu")[5])

In [ ]:
source = ZarrWellDatasetSource(
    "../../data/datasets",
    "euler_2d",
    "test",
    exclude_filters=[
        "grf"
        # "step",
    ],
    window_size=41,
    preload_into_ram=True,
)

In [ ]:
from einops import rearrange


data_np = rearrange(source.preload_cache[0], "b t ... c -> b t c ...")
data_np.shape

In [ ]:
from pathlib import Path

import numpy as np


def sample_crop_starts(n_traj, n_t, window, num_crops, seed=0):
    """Random (traj_idx, t0) pairs, no duplicates, sorted by (traj, t0).
    Stratified (equal count per trajectory) when num_crops is a multiple of n_traj."""
    n_win = n_t - window + 1
    rng = np.random.default_rng(seed)
    if num_crops % n_traj == 0:
        per = num_crops // n_traj
        assert per <= n_win, f"at most {n_win} distinct crops per trajectory"
        t0 = np.stack([rng.choice(n_win, per, replace=False) for _ in range(n_traj)])
        traj_idx = np.repeat(np.arange(n_traj), per)
        t0 = np.sort(t0, axis=1).reshape(-1)
    else:
        flat = np.sort(rng.choice(n_traj * n_win, num_crops, replace=False))
        traj_idx, t0 = np.divmod(flat, n_win)
    return traj_idx.astype(np.int64), t0.astype(np.int64)


def make_crops(data, traj_idx, t0, window, out_path=None):
    """data: (n_traj, T, C, H, W) numpy/zarr/memmap. Returns (num_crops, window, C, H, W).
    Reads each trajectory once. If out_path is given, writes a .npy memmap on disk."""
    shape = (len(t0), window, *data.shape[2:])
    out = (
        np.lib.format.open_memmap(out_path, mode="w+", dtype=data.dtype, shape=shape)
        if out_path is not None
        else np.empty(shape, dtype=data.dtype)
    )
    offsets = np.arange(window)
    trajs, starts_at = np.unique(traj_idx, return_index=True)  # traj_idx is sorted
    bounds = np.append(starts_at, len(traj_idx))
    for i, lo, hi in zip(trajs, bounds[:-1], bounds[1:]):
        traj = np.asarray(data[i])  # one read per trajectory, (T, C, H, W)
        out[lo:hi] = traj[t0[lo:hi, None] + offsets]  # (k, window, C, H, W)
    if out_path is not None:
        out.flush()
    return out


# --- Build once ---
WINDOW, NUM_CROPS = 41, 10000
eval_dir = Path("../../data/eval_crops/euler_2d")
eval_dir.mkdir(parents=True, exist_ok=True)

traj_idx, t0 = sample_crop_starts(*data_np.shape[:2], WINDOW, NUM_CROPS, seed=0)
# np.savez(eval_dir / "crop_index.npz", traj_idx=traj_idx, t0=t0)
crops = make_crops(data_np, traj_idx, t0, WINDOW)  # , out_path=eval_dir / "crops.npy")


def get_loss_args(source):
    dt = source.metadata_common["temporal_resolution"]
    dxs = source.metadata_common["spatial_resolution"]
    return (dt, *dxs)


@eqx.filter_jit
def compute_metrics(model, u, args, context_length=20):
    context, u_data = u[:context_length], u[context_length:]

    u_pred = model.rollout(context, args, num_steps=len(u_data))[0]

    return {
        "l2_onestep": relative_L2_error(u_pred[0], u_data[0]),
        "l_inf_onestep": relative_L_infty_error(u_pred[0], u_data[0]),
        "l2_rollout": relative_L2_error(u_pred, u_data),
        "l_inf_rollout": relative_L_infty_error(u_pred, u_data),
    }


In [ ]:
@eqx.filter_jit
def evaluate(model, crops, args, batch_size=100):
    def one(u):
        return compute_metrics(model, u, args)

    return jax.lax.map(one, crops, batch_size=batch_size)


In [ ]:
crops.shape

In [ ]:
@eqx.filter_jit
def metrics_long_rollout(model, u, args, context_length=20):
    context, u_data = u[:context_length], u[context_length:]
    u_pred = model.rollout(context, args, num_steps=len(u_data))[0]
    l2 = jax.vmap(relative_L2_error)(u_pred, u_data)
    l_inf = jax.vmap(relative_L_infty_error)(u_pred, u_data)
    return {"l2": l2, "l_inf": l_inf}


In [ ]:
checkpoint_dir = Path("../../checkpoints/euler_2d/")

model_paths = {
    "DPOT": [
        "seed=0/26-09-22-02_45_52",
        "seed=10/26-09-22-09_38_04",
        "seed=20/26-09-22-09_38_15",
    ],
    # "HyperFluxFNOLocal": "seed=10/26-04-30-01:34:48",
    # "DISCO": "seed=0/26-07-18-20:51:57",
    "HyperNeuralOperator": [
        "seed=0/26-09-22-13_04_48",
        "seed=10/26-09-23-17_31_47",
        "seed=20/26-09-24-23_34_31",
    ],
    "DISCO": [
        "seed=0/26-09-22-02_32_26",
        "seed=10/26-09-24-01_14_36",
        "seed=20/26-09-24-01_22_00",
    ],
}

In [ ]:
crops.shape

In [ ]:
CHUNK = 500


def evaluate_all(model, crops_host, args, chunk=CHUNK, batch_size=20):
    outs = []
    for lo in range(0, len(crops_host), chunk):
        crops_dev = jnp.asarray(
            crops_host[lo : lo + chunk]
        )  # host -> device, one chunk
        outs.append(jax.device_get(evaluate(model, crops_dev, args, batch_size)))
        del crops_dev
    return jax.tree.map(lambda *xs: np.concatenate(xs, 0), *outs)


In [ ]:
crops.shape

In [ ]:
loss_args = get_loss_args(source)

results_dict = defaultdict(list)
for model_type in (
    "DISCO",
    "HyperNeuralOperator",
    "DPOT",
):
    for p in tqdm(model_paths[model_type]):
        model = load_model(checkpoint_dir / model_type / "OneStepLoss" / p)
        model = eqx.nn.inference_mode(model, True)
        results = evaluate_all(model, crops, loss_args, chunk=CHUNK, batch_size=100)
        print(jax.tree.map(lambda x: x.shape, results))
        results_dict[model_type].append(jax.tree.map(jnp.mean, results))

In [ ]:
results_dict

In [ ]:
import json


with open("../../data/analysis/euler_2d_ood_step.json", "w") as f:
    json.dump(jax.tree.map(lambda x: float(x), results_dict), f)

In [ ]:
for model_name, res in results_dict.items():
    res_ = jax.tree.transpose(jax.tree.structure(["*"] * 3), None, res)
    stats = jax.tree.map(
        lambda list_: {
            "mean": jnp.mean(jnp.asarray(list_)),
            "std": jnp.std(jnp.asarray(list_)),
        },
        res_,
        is_leaf=lambda x: isinstance(x, list),
    )
    for k, v in stats.items():
        print(model_name, f"{k}: {v['mean']:.2e}" + "±" + f"{v['std']:.2e}")

In [ ]:
source.datapaths

In [ ]:
import equinox as eqx
import jax
import jax.numpy as jnp
import numpy as np


# data_np: (1000, 101, 4, 128, 128) numpy / zarr / memmap array, kept on the host
N_TRAJ, N_T = data_np.shape[:2]
WINDOW, CONTEXT = 41, 20
N_WIN = N_T - WINDOW + 1  # 61 valid start times per trajectory
PER_TRAJ = 10  # 10 x 1000 = 10_000 windows
TRAJ_CHUNK = 250  # trajectories on the GPU at once (~6.6 GB fp32); must divide N_TRAJ
MAP_BATCH = 50  # windows per vmapped call inside lax.map (your known-safe batch)

# --- Sample start times once (same windows for every model/seed) ---
rng = np.random.default_rng(0)
t0 = np.stack([rng.choice(N_WIN, PER_TRAJ, replace=False) for _ in range(N_TRAJ)])
t0 = np.sort(t0, axis=1).astype(np.int32)

loss_args = get_loss_args(source)


@eqx.filter_jit
def chunk_metrics(model, data, t0, args):
    """data: (n_traj, T, C, H, W) on device; t0: (n_traj, PER_TRAJ) start indices."""
    n_traj, per_traj = t0.shape
    traj_idx = jnp.repeat(jnp.arange(n_traj, dtype=jnp.int32), per_traj)
    starts = t0.reshape(-1)

    def one(i_s):
        i, s = i_s
        u = jax.lax.dynamic_slice(data, (i, s, 0, 0, 0), (1, WINDOW, *data.shape[2:]))[
            0
        ]
        return eqx.filter_vmap(metrics_long_rollout, in_axes=(None, 0, None, None))(
            model, u, args, CONTEXT
        )

    out = jax.lax.map(one, (traj_idx, starts), batch_size=MAP_BATCH)
    return jax.tree.map(lambda x: x.reshape(n_traj, per_traj, *x.shape[1:]), out)


def evaluate(model):
    outs = []
    for c in range(0, N_TRAJ, TRAJ_CHUNK):
        data = jax.device_put(np.asarray(data_np[c : c + TRAJ_CHUNK]))
        out = chunk_metrics(model, data, jnp.asarray(t0[c : c + TRAJ_CHUNK]), loss_args)
        outs.append(jax.device_get(out))
        del data
    return jax.tree.map(lambda *xs: np.concatenate(xs, axis=0), *outs)

In [ ]:
results_dict = defaultdict(list)
for model_type in ("DISCO", "DPOT", "HyperNeuralOperator"):
    for p in model_paths[model_type]:
        model = eqx.nn.inference_mode(
            load_model(checkpoint_dir / model_type / "OneStepLoss" / p), True
        )
        results_dict[model_type].append(evaluate(model))

In [ ]:
n_batches = 50
dataloader = (
    grain.MapDataset.source(source)
    .seed(0)
    .shuffle()
    .batch(batch_size=200, drop_remainder=False)
)
batches = np.concatenate([b for b in tqdm(itertools.islice(dataloader, n_batches))])
batches.shape

In [ ]:
checkpoint_dir = Path("../../checkpoints/euler_2d/")

model_paths = {
    "DPOT": [
        "seed=0/26-09-22-02_45_52",
        "seed=10/26-09-22-09_38_04",
        "seed=20/26-09-22-09_38_15",
    ],
    # "HyperFluxFNOLocal": "seed=10/26-04-30-01:34:48",
    # "DISCO": "seed=0/26-07-18-20:51:57",
    "HyperNeuralOperator": [
        "seed=0/26-09-22-13_04_48",
        "seed=10/26-09-23-17_31_47",
        "seed=20/26-09-24-23_34_31",
    ],
    "DISCO": [
        "seed=0/26-09-22-02_32_26",
        "seed=10/26-09-24-01_14_36",
        "seed=20/26-09-24-01_22_00",
    ],
}

In [ ]:
source = ZarrWellDatasetSource(
    "../../data/datasets",
    "euler_2d",
    "test",
    exclude_filters=[
        "step",
    ],
    window_size=41,
)

In [ ]:
@eqx.filter_jit
def batched_metrics(model, batch, args):
    return jax.vmap(lambda u: metrics_long_rollout(model, u, args))(batch)


loss_args = get_loss_args(source)  # compute once, outside the loop


In [ ]:
results_dict = defaultdict(list)

for model_type in ("DPOT", "HyperNeuralOperator", "DISCO"):
    for p in model_paths[model_type]:
        model = load_model(checkpoint_dir / model_type / "OneStepLoss" / p)
        model = eqx.nn.inference_mode(model, True)

        res = [batched_metrics(model, batch, loss_args) for batch in tqdm(dataloader)]
        res_ = jax.tree.map(lambda *xs: jnp.stack(xs, axis=0), *res)
        results_dict[model_type].append(res_)

In [ ]:
dataloader = grain.MapDataset.source(source).batch(batch_size=200, drop_remainder=False)

## ID long term predictions

In [ ]:
source = ZarrWellDatasetSource(
    "../../data/datasets",
    "euler_2d",
    "test",
    exclude_filters=[
        "step",
    ],
    window_size=101,
    # preload_into_ram=True,
)
dataloader = grain.MapDataset.source(source).batch(batch_size=200, drop_remainder=False)


In [ ]:
len(source)

In [ ]:
def get_loss_args(source):
    dt = source.metadata_common["temporal_resolution"]
    dxs = source.metadata_common["spatial_resolution"]
    return (dt, *dxs)

In [ ]:
results_dict = defaultdict(list)

for model_type in ("DPOT", "HyperNeuralOperator", "DISCO"):
    for p in model_paths[model_type]:
        model = load_model(checkpoint_dir / model_type / "OneStepLoss" / p)
        model = eqx.nn.inference_mode(model, True)

        res = []
        for batch in tqdm(iter(dataloader)):
            res_batch = eqx.filter_jit(
                eqx.filter_vmap(
                    lambda data: metrics_long_rollout(
                        model, data, get_loss_args(source)
                    )
                )
            )(batch)
            res.append(res_batch)
        res_ = jax.tree.map(lambda *xs: jnp.stack(xs, axis=0), *res)
        results_dict[model_type].append(res_)


In [ ]:
results_dict["DPOT"][0]["l2"].shape

In [ ]:
results_stacked = {
    k: jax.tree.transpose(
        jax.tree.structure(["*"] * 3),
        None,
        v,
    )
    for k, v in results_dict.items()
}
results_stacked = jax.tree.map(
    jnp.stack, results_stacked, is_leaf=lambda x: isinstance(x, list)
)

In [ ]:
from einops import rearrange


results_final = jax.tree.map(
    lambda x: rearrange(x, "seed n_batch batch time -> seed (n_batch batch) time"),
    results_stacked,
)
results_final["DPOT"]["l2"].shape

In [ ]:
import h5py


with h5py.File("../../data/analysis/euler_2d_long_rollout.hdf5", "w") as f:
    for model_name, res_dict in results_final.items():
        if model_name == "HyperNeuralOperator":
            model_name = "HyperFluxFNOLocal"
        g = f.create_group(model_name)
        for metric_name, val in res_dict.items():
            g.create_dataset(metric_name, data=val)

In [ ]:
u_pred_dict = {}

for model_type, path in tqdm(model_paths.items()):
    model = load_model(checkpoint_dir / model_type / "OneStepLoss" / path)
    model = eqx.nn.inference_mode(model, True)

    context, u_data = batch[:, :20], batch[:, 20:]
    u_pred = eqx.filter_vmap(
        lambda x, y: model.rollout(x, get_loss_args(source), num_steps=len(y))[0]
    )(context, u_data)

    u_pred_dict[model_type] = u_pred

In [ ]:
import jax.numpy as jnp
from context_flux_no.metrics import relative_L2_error


err_time_dict = jax.tree.map(
    lambda x: jnp.mean(
        eqx.filter_vmap(eqx.filter_vmap(relative_L2_error))(x[:,], u_data[:,]), axis=0
    ),
    u_pred_dict,
)